# Flight Distance Calculation

This notebook calculates the great circle distance between departure and arrival airports for flight data using the Haversine formula.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
from pathlib import Path
import math
from typing import Tuple, Optional

print("✅ Libraries imported successfully")

In [ ]:
# Load airport coordinates data
data_path = Path("../data")
airport_coords_df = pd.read_csv(data_path / "airport_coordinates.csv")

print(f"✅ Airport coordinates loaded: {len(airport_coords_df)} airports")
print(f"📊 Airports with coordinates: {len(airport_coords_df[airport_coords_df['latitude'].notna()])}")
print(f"\n🔍 First few airports:")
print(airport_coords_df.head())

In [ ]:
# Create airport coordinates lookup dictionary for fast access
airport_coords = {}
valid_coords = airport_coords_df[airport_coords_df['latitude'].notna()]

for _, row in valid_coords.iterrows():
    airport_coords[row['iata_code']] = {
        'latitude': row['latitude'],
        'longitude': row['longitude']
    }

print(f"✅ Airport coordinates lookup created for {len(airport_coords)} airports")
print(f"🔍 Sample lookup: OSL -> {airport_coords.get('OSL', 'Not found')}")

In [ ]:
def haversine_distance(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Calculate the great circle distance between two points on Earth using the Haversine formula.
    
    Args:
        lat1, lon1: Latitude and longitude of first point in decimal degrees
        lat2, lon2: Latitude and longitude of second point in decimal degrees
        
    Returns:
        Distance in kilometers
    """
    # Convert latitude and longitude from degrees to radians
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    c = 2 * math.asin(math.sqrt(a))
    
    # Radius of Earth in kilometers
    R = 6371.0
    
    # Calculate the distance
    distance = R * c
    
    return distance

# Test the function
print("🧪 Testing Haversine distance function:")
# Oslo (OSL) to Bergen (BGO) - should be around 300km
osl_coords = airport_coords.get('OSL')
bgo_coords = airport_coords.get('BGO')

if osl_coords and bgo_coords:
    test_distance = haversine_distance(
        osl_coords['latitude'], osl_coords['longitude'],
        bgo_coords['latitude'], bgo_coords['longitude']
    )
    print(f"✅ OSL to BGO distance: {test_distance:.2f} km")
else:
    print("❌ Could not find coordinates for OSL or BGO")

print("✅ Haversine distance function created")

In [ ]:
def calculate_flight_distance(dep_airport: str, arr_airport: str) -> Optional[float]:
    """
    Calculate the distance between two airports using their IATA codes.
    
    Args:
        dep_airport: Departure airport IATA code
        arr_airport: Arrival airport IATA code
        
    Returns:
        Distance in kilometers, or None if coordinates not available
    """
    # Get coordinates for both airports
    dep_coords = airport_coords.get(dep_airport)
    arr_coords = airport_coords.get(arr_airport)
    
    # Check if both airports have coordinates
    if not dep_coords or not arr_coords:
        return None
    
    # Calculate distance
    distance = haversine_distance(
        dep_coords['latitude'], dep_coords['longitude'],
        arr_coords['latitude'], arr_coords['longitude']
    )
    
    return distance

# Test the function
print("🧪 Testing flight distance calculation:")
test_routes = [
    ('OSL', 'BGO'),  # Oslo to Bergen
    ('OSL', 'TRD'),  # Oslo to Trondheim
    ('OSL', 'CPH'),  # Oslo to Copenhagen
    ('OSL', 'ARN'),  # Oslo to Stockholm
]

for dep, arr in test_routes:
    distance = calculate_flight_distance(dep, arr)
    if distance:
        print(f"✅ {dep} → {arr}: {distance:.2f} km")
    else:
        print(f"❌ {dep} → {arr}: Could not calculate distance")

print("\n✅ Flight distance calculation function created")

In [ ]:
# Load historical flights data
print("📊 Loading historical flights data...")
df_flights = pd.read_csv(data_path / "avinor_data/historical_flights.csv")

print(f"✅ Historical flights loaded: {len(df_flights):,} flights")
print(f"📋 Columns: {list(df_flights.columns)}")
print(f"\n🔍 Sample data:")
print(df_flights[['flight_id', 'dep_airport', 'arr_airport']].head())

In [ ]:
def add_distance_to_flights(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add distance calculations to flight data.
    
    Args:
        df: DataFrame with 'dep_airport' and 'arr_airport' columns
        
    Returns:
        DataFrame with added 'distance_km' column
    """
    print("🧮 Calculating distances for all flights...")
    
    # Create a copy to avoid modifying original
    df_with_distance = df.copy()
    
    # Calculate distances
    distances = []
    missing_coords_count = 0
    
    for idx, row in df_with_distance.iterrows():
        if idx % 50000 == 0:  # Progress indicator
            print(f"   📍 Processed {idx:,} flights...")
            
        distance = calculate_flight_distance(row['dep_airport'], row['arr_airport'])
        distances.append(distance)
        
        if distance is None:
            missing_coords_count += 1
    
    # Add distance column
    df_with_distance['distance_km'] = distances
    
    # Summary statistics
    valid_distances = df_with_distance['distance_km'].dropna()
    
    print(f"\n📊 DISTANCE CALCULATION SUMMARY:")
    print(f"   • Total flights: {len(df_with_distance):,}")
    print(f"   • Distances calculated: {len(valid_distances):,}")
    print(f"   • Missing coordinates: {missing_coords_count:,}")
    print(f"   • Success rate: {(len(valid_distances)/len(df_with_distance)*100):.1f}%")
    
    if len(valid_distances) > 0:
        print(f"\n🌍 DISTANCE STATISTICS:")
        print(f"   • Average distance: {valid_distances.mean():.2f} km")
        print(f"   • Median distance: {valid_distances.median():.2f} km")
        print(f"   • Min distance: {valid_distances.min():.2f} km")
        print(f"   • Max distance: {valid_distances.max():.2f} km")
        print(f"   • Standard deviation: {valid_distances.std():.2f} km")
    
    return df_with_distance

print("✅ Distance calculation function for datasets created")

In [ ]:
# Calculate distances for the entire dataset
print("🚀 Starting distance calculation for all historical flights...")
print("This may take a few moments...\n")

df_flights_with_distance = add_distance_to_flights(df_flights)

print("\n✅ Distance calculation completed!")

In [ ]:
# Analyze the results
print("📈 DETAILED ANALYSIS OF FLIGHT DISTANCES:")
print("=" * 50)

# Show sample results
print("\n🔍 Sample flights with calculated distances:")
sample_with_distance = df_flights_with_distance[['flight_id', 'dep_airport', 'arr_airport', 'distance_km']].dropna().head(10)
print(sample_with_distance.to_string(index=False))

# Show flights without distance (missing coordinates)
missing_distance = df_flights_with_distance[df_flights_with_distance['distance_km'].isna()]
if len(missing_distance) > 0:
    print(f"\n❌ Flights without calculated distance ({len(missing_distance)} total):")
    unique_missing_routes = missing_distance[['dep_airport', 'arr_airport']].drop_duplicates().head(10)
    print(unique_missing_routes.to_string(index=False))
    if len(unique_missing_routes) < len(missing_distance[['dep_airport', 'arr_airport']].drop_duplicates()):
        remaining = len(missing_distance[['dep_airport', 'arr_airport']].drop_duplicates()) - len(unique_missing_routes)
        print(f"   ... and {remaining} more unique routes")

# Distance distribution analysis
valid_distances = df_flights_with_distance['distance_km'].dropna()
if len(valid_distances) > 0:
    print(f"\n📊 DISTANCE DISTRIBUTION:")
    
    # Define distance categories
    categories = [
        (0, 500, "Short-haul (< 500 km)"),
        (500, 1500, "Medium-haul (500-1500 km)"),
        (1500, 4000, "Long-haul (1500-4000 km)"),
        (4000, float('inf'), "Ultra long-haul (> 4000 km)")
    ]
    
    for min_dist, max_dist, category_name in categories:
        count = len(valid_distances[(valid_distances >= min_dist) & (valid_distances < max_dist)])
        percentage = (count / len(valid_distances)) * 100
        print(f"   • {category_name}: {count:,} flights ({percentage:.1f}%)")
    
    # Top 10 longest routes
    print(f"\n🌍 TOP 10 LONGEST ROUTES:")
    longest_routes = df_flights_with_distance.nlargest(10, 'distance_km')[['dep_airport', 'arr_airport', 'distance_km']]
    for _, row in longest_routes.iterrows():
        print(f"   • {row['dep_airport']} → {row['arr_airport']}: {row['distance_km']:.0f} km")
    
    # Top 10 shortest routes (excluding 0 distance)
    print(f"\n🏃 TOP 10 SHORTEST ROUTES:")
    shortest_routes = df_flights_with_distance[df_flights_with_distance['distance_km'] > 0].nsmallest(10, 'distance_km')[['dep_airport', 'arr_airport', 'distance_km']]
    for _, row in shortest_routes.iterrows():
        print(f"   • {row['dep_airport']} → {row['arr_airport']}: {row['distance_km']:.0f} km")

In [ ]:
# Save the enhanced dataset with distances
print("💾 Saving flight data with calculated distances...")

# Save to new CSV file
output_file = data_path / "historical_flights_with_distance.csv"
df_flights_with_distance.to_csv(output_file, index=False)

print(f"✅ Enhanced dataset saved to: {output_file}")
print(f"📊 Dataset contains {len(df_flights_with_distance):,} flights with distance calculations")

# Show final summary
print(f"\n🎯 FINAL SUMMARY:")
print(f"   • Original flights: {len(df_flights):,}")
print(f"   • Flights with distance: {len(df_flights_with_distance[df_flights_with_distance['distance_km'].notna()]):,}")
print(f"   • Success rate: {(len(df_flights_with_distance[df_flights_with_distance['distance_km'].notna()])/len(df_flights)*100):.1f}%")
print(f"   • New column added: 'distance_km'")
print(f"   • Data ready for analysis and modeling")

## Usage Examples

The distance calculation functionality can be used in several ways:

### 1. Calculate distance for individual flight routes:
```python
distance = calculate_flight_distance('OSL', 'BGO')
print(f"Distance: {distance:.2f} km")
```

### 2. Add distances to any flight dataset:
```python
# For any DataFrame with 'dep_airport' and 'arr_airport' columns
df_with_distances = add_distance_to_flights(your_flight_data)
```

### 3. Use the Haversine formula directly:
```python
# For any two coordinate pairs
distance = haversine_distance(lat1, lon1, lat2, lon2)
```

The distance calculations use the great circle distance (shortest distance over Earth's surface) and are accurate for aviation purposes.